In [1]:
# IMPORT PACKAGES

import os
from pathlib import Path

import geopandas as gpd
import rasterio
from rasterio.mask import mask
from tqdm import tqdm

In [2]:

# ROOT DIRECTORY

ROOT_DIR = Path("..")

# INPUT DATASET
DATASET_DIR = ROOT_DIR / "dataset"

# OUTPUT CLIPPED DATASET
CLIPPED_DIR = ROOT_DIR / "dataset_clipped"

# AOI FILE
AOI_PATH = ROOT_DIR / "aoi" / "field_boundary.gpkg"

# Create output folder
CLIPPED_DIR.mkdir(exist_ok=True)

print("Paths ready.")


Paths ready.


In [3]:
# LOAD AOI

aoi = gpd.read_file(AOI_PATH)

print(aoi)

print("\nAOI loaded correctly.")

                                            geometry
0  POLYGON ((354279.208 5789622.602, 354250.544 5...

AOI loaded correctly.


In [4]:
def clip_rasters():

    # Find years
    year_folders = sorted([
        f for f in DATASET_DIR.iterdir()
        if f.is_dir()
    ])

    for year_folder in year_folders:

        year_name = year_folder.name

        print(f"\nPROCESSING YEAR: {year_name}")

        # Create output year folder
        output_year_folder = CLIPPED_DIR / year_name
        output_year_folder.mkdir(exist_ok=True)

        # Find flights
        flight_folders = sorted([
            f for f in year_folder.iterdir()
            if f.is_dir()
        ])

        for flight_folder in tqdm(flight_folders):

            flight_name = flight_folder.name

            # Output flight folder
            output_flight_folder = output_year_folder / flight_name
            output_flight_folder.mkdir(exist_ok=True)

            # -----------------------------------------
            # CHECK IF ALREADY PROCESSED
            # -----------------------------------------

            existing_tifs = list(
                output_flight_folder.glob("*.tif")
            )

            if len(existing_tifs) > 0:

                print(
                    f"\n{flight_name} already processed. Skipping."
                )

                continue

            # Find tif files
            tif_files = list(flight_folder.glob("*.tif"))

            print(f"\n{flight_name} -> {len(tif_files)} tif files")

            for tif_path in tif_files:

                output_path = output_flight_folder / tif_path.name

                # Skip if already exists
                if output_path.exists():

                    print(
                        f"Skipping existing file: {tif_path.name}"
                    )

                    continue

                try:

                    with rasterio.open(tif_path) as src:

                        # Reproject AOI if CRS differs
                        if aoi.crs != src.crs:
                            aoi_projected = aoi.to_crs(src.crs)
                        else:
                            aoi_projected = aoi

                        # Extract geometry
                        geometries = (
                            aoi_projected.geometry.values
                        )

                        # Clip raster
                        clipped_image, clipped_transform = mask(
                            src,
                            geometries,
                            crop=True
                        )

                        # Copy metadata
                        clipped_meta = src.meta.copy()

                        # Update metadata
                        clipped_meta.update({
                            "height": clipped_image.shape[1],
                            "width": clipped_image.shape[2],
                            "transform": clipped_transform
                        })

                        # Save clipped raster
                        with rasterio.open(
                            output_path,
                            "w",
                            **clipped_meta
                        ) as dest:

                            dest.write(clipped_image)

                    print(f"Clipped: {tif_path.name}")

                except Exception as e:

                    print(f"ERROR with {tif_path.name}")
                    print(e)

    print("\nALL CLIPPING COMPLETED.")

In [5]:
clip_rasters()


PROCESSING YEAR: 2025


100%|█████████████████████████████████████████████████████████████████████████████████| 11/11 [00:00<00:00, 678.81it/s]



flight_2025_04_09 already processed. Skipping.

flight_2025_05_08 already processed. Skipping.

flight_2025_05_14 already processed. Skipping.

flight_2025_05_19 already processed. Skipping.

flight_2025_06_03 already processed. Skipping.

flight_2025_06_12 already processed. Skipping.

flight_2025_06_19 already processed. Skipping.

flight_2025_07_04 already processed. Skipping.

flight_2025_07_08 already processed. Skipping.

flight_2025_08_11 already processed. Skipping.

flight_2025_08_19 already processed. Skipping.

PROCESSING YEAR: 2026


  0%|                                                                                            | 0/9 [00:00<?, ?it/s]


flight_2026_04_01 already processed. Skipping.

flight_2026_04_07 already processed. Skipping.

flight_2026_04_15 already processed. Skipping.

flight_2026_04_22 already processed. Skipping.

flight_2026_05_01 already processed. Skipping.

flight_2026_05_22 already processed. Skipping.

flight_2026_05_28 -> 6 tif files
Clipped: 2026_05_28_multispectral_transparent_reflectance_green.tif
Clipped: 2026_05_28_multispectral_transparent_reflectance_group1.tif
Clipped: 2026_05_28_multispectral_transparent_reflectance_nir.tif
Clipped: 2026_05_28_multispectral_transparent_reflectance_red edge.tif


...

Clipped: 2026_05_28_multispectral_transparent_reflectance_red.tif
Clipped: 2026_05_28_thermal_transparent_reflectance_grayscale.tif

flight_2026_06_01 -> 6 tif files
Clipped: 2026_06_01_multispectral_transparent_reflectance_green.tif
Clipped: 2026_06_01_multispectral_transparent_reflectance_group1.tif
Clipped: 2026_06_01_multispectral_transparent_reflectance_nir.tif
Clipped: 2026_06_01_multispectral_transparent_reflectance_red edge.tif


 89%|██████████████████████████████████████████████████████████████████████████▋         | 8/9 [00:13<00:01,  1.93s/it]

Clipped: 2026_06_01_multispectral_transparent_reflectance_red.tif
Clipped: 2026_06_01_thermal_transparent_reflectance_grayscale.tif

flight_2026_06_19 -> 6 tif files
Clipped: 2026_06_19_multispectral_transparent_reflectance_green.tif
Clipped: 2026_06_19_multispectral_transparent_reflectance_group1.tif
Clipped: 2026_06_19_multispectral_transparent_reflectance_nir.tif
Clipped: 2026_06_19_multispectral_transparent_reflectance_red edge.tif


100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:20<00:00,  2.31s/it]

Clipped: 2026_06_19_multispectral_transparent_reflectance_red.tif
Clipped: 2026_06_19_thermal_transparent_reflectance_grayscale.tif

ALL CLIPPING COMPLETED.
